In [4]:
import pandas as pd

file_path = "returns.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset loaded successfully!
Shape: (1641, 8)

Columns:
['return_id', 'order_id', 'customer_id', 'product_id', 'return_date', 'return_reason', 'refund_amount', 'return_status']


In [5]:
df_original = df.copy()

print("Backup created successfully!")
print("Original shape:", df_original.shape)
print("Current shape:", df.shape)

Backup created successfully!
Original shape: (1641, 8)
Current shape: (1641, 8)


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1641 entries, 0 to 1640
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   return_id      1641 non-null   str    
 1   order_id       1641 non-null   str    
 2   customer_id    1641 non-null   str    
 3   product_id     1641 non-null   str    
 4   return_date    1641 non-null   str    
 5   return_reason  1641 non-null   str    
 6   refund_amount  1641 non-null   float64
 7   return_status  1641 non-null   str    
dtypes: float64(1), str(7)
memory usage: 222.5 KB


In [7]:
missing_report = df.isna().sum()

print("Missing values in each column:")
print(missing_report)

print("\nTotal missing values:", missing_report.sum())

Missing values in each column:
return_id        0
order_id         0
customer_id      0
product_id       0
return_date      0
return_reason    0
refund_amount    0
return_status    0
dtype: int64

Total missing values: 0


In [8]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


In [9]:
id_columns = ["return_id", "order_id", "customer_id", "product_id"]

for col in id_columns:
    print(f"\n{col}")
    print("Missing:", df[col].isna().sum())
    print("Unique:", df[col].nunique())
    print("Repeated values:", df[col].duplicated().sum())


return_id
Missing: 0
Unique: 1641
Repeated values: 0

order_id
Missing: 0
Unique: 1641
Repeated values: 0

customer_id
Missing: 0
Unique: 1582
Repeated values: 59

product_id
Missing: 0
Unique: 367
Repeated values: 1274


In [10]:
print("Invalid return_id prefix:",
      (~df["return_id"].str.startswith("RET")).sum())

print("Invalid order_id prefix:",
      (~df["order_id"].str.startswith("ORD")).sum())

print("Invalid customer_id prefix:",
      (~df["customer_id"].str.startswith("CUST")).sum())

print("Invalid product_id prefix:",
      (~df["product_id"].str.startswith("PROD")).sum())

Invalid return_id prefix: 0
Invalid order_id prefix: 0
Invalid customer_id prefix: 0
Invalid product_id prefix: 0


In [11]:
df["return_date"] = pd.to_datetime(
    df["return_date"],
    errors="coerce"
)

print("Data type:", df["return_date"].dtype)

print("Invalid/missing return dates:",
      df["return_date"].isna().sum())

print("Earliest return date:",
      df["return_date"].min())

print("Latest return date:",
      df["return_date"].max())

Data type: datetime64[us]
Invalid/missing return dates: 0
Earliest return date: 2023-01-15 00:00:00
Latest return date: 2025-12-31 00:00:00


In [12]:
print("Return Reasons:")
print(df["return_reason"].value_counts(dropna=False))

print("\nReturn Status:")
print(df["return_status"].value_counts(dropna=False))

Return Reasons:
return_reason
Size issue                 312
Changed mind               275
Product damaged            257
Quality issue              242
Wrong product              178
Product not as expected    167
Late delivery              147
Other                       63
Name: count, dtype: int64

Return Status:
return_status
Refund Completed    1147
Approved             245
Pending              145
Rejected             104
Name: count, dtype: int64


In [13]:
print("Refund amount summary:")
print(df["refund_amount"].describe())

print("\nNegative refund amounts:",
      (df["refund_amount"] < 0).sum())

print("Zero refund amounts:",
      (df["refund_amount"] == 0).sum())

print("\nRefund amount by return status:")
print(
    df.groupby("return_status")["refund_amount"]
      .agg(["count", "min", "max", "mean", "sum"])
)

Refund amount summary:
count      1641.000000
mean       8602.532145
std       25123.193733
min           0.000000
25%         453.100000
50%        1725.000000
75%        5436.000000
max      330250.000000
Name: refund_amount, dtype: float64

Negative refund amounts: 0
Zero refund amounts: 249

Refund amount by return status:
                  count   min       max          mean          sum
return_status                                                     
Approved            245  92.0  330250.0  11727.413469   2873216.30
Pending             145   0.0       0.0      0.000000         0.00
Refund Completed   1147  41.0  297225.0   9802.562293  11243538.95
Rejected            104   0.0       0.0      0.000000         0.00


In [14]:

approved_zero = (
    (df["return_status"] == "Approved") &
    (df["refund_amount"] == 0)
).sum()

completed_zero = (
    (df["return_status"] == "Refund Completed") &
    (df["refund_amount"] == 0)
).sum()

pending_positive = (
    (df["return_status"] == "Pending") &
    (df["refund_amount"] > 0)
).sum()

rejected_positive = (
    (df["return_status"] == "Rejected") &
    (df["refund_amount"] > 0)
).sum()

print("Approved with zero refund:", approved_zero)
print("Refund Completed with zero refund:", completed_zero)
print("Pending with positive refund:", pending_positive)
print("Rejected with positive refund:", rejected_positive)

Approved with zero refund: 0
Refund Completed with zero refund: 0
Pending with positive refund: 0
Rejected with positive refund: 0


In [15]:
print("Refund amount by return reason:")

print(
    df.groupby("return_reason")["refund_amount"]
      .agg(["count", "min", "max", "mean", "sum"])
      .sort_values("sum", ascending=False)
)

Refund amount by return reason:
                         count  min        max          mean         sum
return_reason                                                           
Quality issue              242  0.0  217783.50  12471.764050  3018166.90
Changed mind               275  0.0  297225.00  10260.227636  2821562.60
Size issue                 312  0.0  125495.00   6191.648558  1931794.35
Product damaged            257  0.0  330250.00   7449.737743  1914582.60
Late delivery              147  0.0  290378.00   9673.265986  1421970.10
Wrong product              178  0.0  148608.60   7988.105056  1421882.70
Product not as expected    167  0.0  175362.75   7681.906287  1282878.35
Other                       63  0.0   62616.45   4824.089683   303917.65


In [16]:

print("Negative refund amounts:",
      (df["refund_amount"] < 0).sum())

print("Zero refund amounts:",
      (df["refund_amount"] == 0).sum())

print("Positive refund amounts:",
      (df["refund_amount"] > 0).sum())

print("\nRefund amount data type:",
      df["refund_amount"].dtype)

print("\nRefund amount range:")
print("Minimum:", df["refund_amount"].min())
print("Maximum:", df["refund_amount"].max())

Negative refund amounts: 0
Zero refund amounts: 249
Positive refund amounts: 1392

Refund amount data type: float64

Refund amount range:
Minimum: 0.0
Maximum: 330250.0


In [17]:

Q1 = df["refund_amount"].quantile(0.25)
Q3 = df["refund_amount"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[
    (df["refund_amount"] < lower_bound) |
    (df["refund_amount"] > upper_bound)
]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Outlier count:", len(outliers))

Q1: 453.1
Q3: 5436.0
IQR: 4982.9
Lower bound: -7021.249999999999
Upper bound: 12910.349999999999
Outlier count: 215


In [18]:

outliers = df[df["refund_amount"] > upper_bound]

print("Number of outliers:", len(outliers))

print("\nOutlier refund amount summary:")
print(outliers["refund_amount"].describe())

print("\nHighest refund records:")
print(
    outliers[
        [
            "return_id",
            "order_id",
            "customer_id",
            "product_id",
            "return_reason",
            "refund_amount",
            "return_status"
        ]
    ]
    .sort_values("refund_amount", ascending=False)
    .head(10)
)

Number of outliers: 215

Outlier refund amount summary:
count       215.000000
mean      50247.550465
std       52730.489082
min       12925.800000
25%       17584.300000
50%       27180.000000
75%       61970.000000
max      330250.000000
Name: refund_amount, dtype: float64

Highest refund records:
      return_id    order_id customer_id product_id    return_reason  \
604   RET107530  ORD1073703  CUST100332  PROD10246  Product damaged   
649   RET108032  ORD1078738  CUST111856  PROD10246     Changed mind   
1243  RET111171  ORD1054854  CUST120664  PROD10058    Late delivery   
1239  RET111153  ORD1053978  CUST124256  PROD10058    Quality issue   
1293  RET111261  ORD1059370  CUST124177  PROD10432  Product damaged   
592   RET107409  ORD1072336  CUST123564  PROD10246     Changed mind   
1348  RET111400  ORD1066824  CUST116135  PROD10246     Changed mind   
976   RET110517  ORD1021255  CUST101193  PROD10246    Quality issue   
1594  RET111963  ORD1094396  CUST122575  PROD10257    Qualit

In [19]:

print("FINAL INTEGRITY CHECK")
print("=" * 50)

print("Original shape:", df_original.shape)
print("Current shape:", df.shape)

print("Rows removed:", len(df_original) - len(df))
print("Columns removed:", len(df_original.columns) - len(df.columns))

print("\nDuplicate rows:", df.duplicated().sum())
print("Total missing values:", df.isna().sum().sum())

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

FINAL INTEGRITY CHECK
Original shape: (1641, 8)
Current shape: (1641, 8)
Rows removed: 0
Columns removed: 0

Duplicate rows: 0
Total missing values: 0

Column names:
['return_id', 'order_id', 'customer_id', 'product_id', 'return_date', 'return_reason', 'refund_amount', 'return_status']

Data types:
return_id                   str
order_id                    str
customer_id                 str
product_id                  str
return_date      datetime64[us]
return_reason               str
refund_amount           float64
return_status               str
dtype: object


In [20]:

output_file = "returns_cleaned.csv"

df.to_csv(output_file, index=False)

print("Saved successfully:", output_file)

Saved successfully: returns_cleaned.csv


In [21]:

import os

print("File exists:", os.path.exists(output_file))

if os.path.exists(output_file):
    print("File size:", os.path.getsize(output_file), "bytes")

    check_df = pd.read_csv(output_file)

    print("Saved file shape:", check_df.shape)
    print("Saved file columns:", check_df.columns.tolist())

File exists: True
File size: 146699 bytes
Saved file shape: (1641, 8)
Saved file columns: ['return_id', 'order_id', 'customer_id', 'product_id', 'return_date', 'return_reason', 'refund_amount', 'return_status']
